In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt

from piano_tiles_ai.separation import separate_piano
from piano_tiles_ai.transcription import transcribe_piano
from piano_tiles_ai.notes import load_notes
from piano_tiles_ai.hands import assign_hands_by_split, MIDDLE_C

piano_path = separate_piano("../data/LetItBe.mp3", cache_dir="../outputs/separated")
midi_path = transcribe_piano(piano_path, cache_dir="../outputs/transcribed")
notes = assign_hands_by_split(load_notes(midi_path))

n_left = sum(n.hand == "left" for n in notes)
print(f"{n_left} left-hand notes, {len(notes) - n_left} right-hand notes")

In [ ]:
OFFSET, DURATION = 60.0, 20.0
COLORS = {"left": "tab:blue", "right": "tab:orange"}

window = [n for n in notes if n.end > OFFSET and n.start < OFFSET + DURATION]

fig, ax = plt.subplots(figsize=(16, 6))
for hand, color in COLORS.items():
    hand_notes = [n for n in window if n.hand == hand]
    ax.hlines(
        y=[n.pitch for n in hand_notes],
        xmin=[n.start for n in hand_notes],
        xmax=[n.end for n in hand_notes],
        colors=color,
        linewidth=4,
        label=f"{hand} hand",
    )
ax.axhline(MIDDLE_C - 0.5, color="gray", linestyle="--", label="split (middle C)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("MIDI pitch")
ax.set_title("Hand assignment baseline: split at middle C")
ax.legend()
plt.show()

In [ ]:
from pathlib import Path

from piano_tiles_ai.evaluation import score_hands
from piano_tiles_ai.hand_data import load_hand_labeled_notes, strip_hands
from piano_tiles_ai.hands import assign_hands_by_split

DATA_DIR = Path("../data/hands/piano-midi-de")

all_predicted, all_reference = [], []
for path in sorted(DATA_DIR.glob("*.mid")):
    try:
        reference = load_hand_labeled_notes(path)
    except ValueError as error:
        print("skipped:", error)
        continue
    predicted = assign_hands_by_split(strip_hands(reference))
    scores = score_hands(predicted, reference)
    print(f"{path.name:30s} {scores.n_notes:6d} notes   accuracy {scores.accuracy:.1%}")
    all_predicted += predicted
    all_reference += reference

print()
print(score_hands(all_predicted, all_reference))

In [ ]:
from piano_tiles_ai.hands import assign_hands_by_clustering, assign_hands_by_split, MIDDLE_C
def evaluate(method, files):
    """Score a hand assignment method on a list of MIDI files."""
    all_predicted, all_reference = [], []
    for path in files:
        try:
            reference = load_hand_labeled_notes(path)
        except ValueError:
            continue
        all_predicted += method(strip_hands(reference))
        all_reference += reference
    return score_hands(all_predicted, all_reference)


files = sorted(DATA_DIR.glob("*.mid"))
print("split      ", evaluate(assign_hands_by_split, files))
print("clustering ", evaluate(assign_hands_by_clustering, files))

In [ ]:
from collections import Counter

from piano_tiles_ai.hand_data import split_for

files = sorted(DATA_DIR.glob("*.mid"))
for path in files:
    print(f"{split_for(path):5s}  {path.name}")
print(Counter(split_for(p) for p in files))

In [ ]:
from piano_tiles_ai.evaluation import evaluate_method
from piano_tiles_ai.hand_data import load_hand_labeled_notes, split_for


def load_pieces(paths):
    """Load the hand-labeled notes of each file, skipping unusable files."""
    pieces = []
    for path in paths:
        try:
            pieces.append(load_hand_labeled_notes(path))
        except ValueError as error:
            print("skipped:", error)
    return pieces


files = sorted(DATA_DIR.glob("*.mid"))
pieces = {
    split: load_pieces([p for p in files if split_for(p) == split])
    for split in ("train", "val", "test")
}

for split, split_pieces in pieces.items():
    notes = [n for piece in split_pieces for n in piece]
    left_share = sum(n.hand == "left" for n in notes) / len(notes)
    print(f"{split:5s}  {len(split_pieces):3d} pieces  {len(notes):7d} notes  {left_share:.0%} left hand")

In [ ]:
from functools import partial
from itertools import product

from piano_tiles_ai.hands import assign_hands_by_clustering, assign_hands_by_split

results = []
for window, min_spread in product([0.5, 1.0, 2.0, 3.0, 5.0], [7, 12, 17]):
    method = partial(assign_hands_by_clustering, window=window, min_spread=min_spread)
    accuracy = evaluate_method(method, pieces["val"]).accuracy
    results.append((accuracy, window, min_spread))
    print(f"window={window:3.1f}s  min_spread={min_spread:2d}  val accuracy={accuracy:.1%}")

best_accuracy, best_window, best_min_spread = max(results)
print(f"\nBest: window={best_window}s, min_spread={best_min_spread} -> {best_accuracy:.1%}")
print(f"Split baseline on val: {evaluate_method(assign_hands_by_split, pieces['val']).accuracy:.1%}")